# Quality Control → Roboflow Delete List

**What this notebook does:** Scans the dataset once and produces `roboflow_delete_list.txt` —
a list grouped by reason, with **filenames as they appear in Roboflow** for pasting directly into name search.

**What it does not do:** It does not delete, modify, or touch Roboflow. Roboflow is the source of truth —
every fix is done there manually, then `Generate` a new version.

| | |
|---|---|
| Where it runs | Locally, VS Code / Jupyter |
| Free space needed | ~1.1GB peak (zip + extracted). The zip is deleted right after extraction. |
| Output | `roboflow_delete_list.txt` — the delete list |
| Runtime | 1–3 minutes (download is most of the time) |

**Workflow:** Run everything → open the txt → delete/fix in Roboflow → `Generate` a new version →
**send a new link to the team** (the old link is a snapshot and does not update on its own) → replace `ROBOFLOW_URL` below and re-run to verify.


## §0 — Settings

Everything you might want to change is here.


In [ ]:
from pathlib import Path

# Export link from Roboflow (Download Dataset > YOLOv8 > show download code > Terminal)
# The link expires after a while — if you get a 404, re-export and paste a fresh link.
ROBOFLOW_URL = "https://app.roboflow.com/ds/lhrziicySt?key=K9FweSTnV6"

# Project URL itself — written at the top of the action file so there is somewhere to click
PROJECT_URL  = "https://app.roboflow.com/chagitvain02-gmail-com/wheelchair-9qvfx-bchvo"

DATASET_DIR = Path("dataset")        # extract destination, relative to the notebook folder
OUT_TXT     = Path("roboflow_delete_list.txt")

EXPECTED_CLASSES = ["person", "wheelchair", "people_wheelchair"]

NEAR_DUP_BITS = 5    # 0 = exact visual identity. 5 = "almost the same image". Higher = more findings.
MIN_SHORT_SIDE = 32  # image whose short side is smaller than this — suspected junk
BOX_EPS = 1e-3       # tolerance for box going outside 0..1. 1e-3 ≈ 0.64px at 640 — export rounding, not a labeling error.

print("Extract destination:", DATASET_DIR.resolve())


## §1 — Download and extract

Checks free space before starting, skips if the dataset already exists, and deletes the zip right after extraction.


In [ ]:
import shutil, urllib.request, zipfile

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def has_images(root: Path) -> bool:
    return root.is_dir() and any(
        p.suffix.lower() in IMG_EXT for p in root.rglob("*") if p.is_file())

if has_images(DATASET_DIR):
    print("✅ Dataset already exists at", DATASET_DIR.resolve(), "— skipping download.")
    print("   (to pull a new version: delete the folder and re-run)")
else:
    free_gb = shutil.disk_usage(Path.cwd().anchor).free / 1024**3
    print(f"Free space: {free_gb:.1f}GB")
    if free_gb < 1.5:
        raise SystemExit(
            f"❌ Only {free_gb:.1f}GB free, need ~1.1GB peak.\n"
            "   Free up space, or run the notebook in Colab (download goes to Google's temp disk)."
        )

    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATASET_DIR / "_roboflow.zip"

    def progress(block, block_size, total):
        done = block * block_size
        pct = f"{100 * done / total:5.1f}%" if total > 0 else "  ?  "
        print(f"\rDownloading... {pct}  ({done / 1024**2:6.1f}MB)", end="")

    print("Downloading from Roboflow (496MB, a few minutes)...")
    urllib.request.urlretrieve(ROBOFLOW_URL, zip_path, reporthook=progress)
    print("\nExtracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASET_DIR)
    zip_path.unlink()          # don't keep 496MB unused
    print("✅ Dataset at", DATASET_DIR.resolve(), " (zip deleted)")


## §2 — Locate splits and check class order

**Why class order is critical:** YOLO does not store names in label files, only numbers. A label line is
`2 0.51 0.44 0.18 0.33` — and the `2` only gets meaning from its position in the list in `data.yaml`.
If the order differs between what you ship and the trainer's code, nothing will fail: training will run,
the numbers will look reasonable, and the model will simply learn swapped classes. A one-second check here saves a day.


In [ ]:
import yaml

ALIASES = {"train": "train", "valid": "valid", "val": "valid",
           "validation": "valid", "test": "test"}

splits = {}
for d in DATASET_DIR.rglob("*"):
    if not d.is_dir():
        continue
    # Roboflow layout: train/images + train/labels   |   alternate: images/train + labels/train
    if d.name == "images" and (key := ALIASES.get(d.parent.name.lower())):
        splits[key] = {"images": d, "labels": d.parent / "labels"}
    elif d.parent.name == "images" and (key := ALIASES.get(d.name.lower())):
        splits[key] = {"images": d, "labels": d.parent.parent / "labels" / d.name}

if not splits:
    raise SystemExit("No split folders found under " + str(DATASET_DIR))

for k in ("train", "valid", "test"):
    if k in splits:
        n = sum(1 for p in splits[k]["images"].iterdir()
                if p.is_file() and p.suffix.lower() in IMG_EXT)
        print(f"  {k:6s}: {n:5d} images   ({splits[k]['images'].relative_to(DATASET_DIR)})")
    else:
        print(f"  {k:6s}:     — missing")

# ── Class order ──────────────────────────────────────────────────────────────
DATA_YAML = sorted(DATASET_DIR.rglob("data.yaml"))[0]
names = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))["names"]
if isinstance(names, dict):
    names = [names[k] for k in sorted(names, key=int)]
CLASS_NAMES = list(names)

print("\nActual order  :", CLASS_NAMES)
print("Expected order:", EXPECTED_CLASSES)
CLASSES_OK = CLASS_NAMES == EXPECTED_CLASSES
print("✅ Match" if CLASSES_OK
      else "🔴 Mismatch! Anyone training must use the order in data.yaml, not the order above.")


## §3 — Scan all images (once)

Opens each file once and collects everything into a table: whether it opened, size, and two fingerprints —

- **MD5** of the bytes: two files with the same MD5 are identical. Finds exact copies, but is blind
  to the tiniest change — re-saving as a different JPEG changes it completely.
- **dHash**: shrinks to 9×8 grayscale and encodes only *whether* each pixel is brighter than the one to its right → 64 bits.
  Two images that look the same get an almost identical hash, even if the files differ. Difference is measured in bits.


In [ ]:
import hashlib
import numpy as np
import pandas as pd
from PIL import Image

def dhash(img, size=8) -> int:
    small = img.convert("L").resize((size + 1, size), Image.Resampling.LANCZOS)
    a = np.asarray(small, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()       # 8x8 = 64 left-right comparisons
    return int.from_bytes(np.packbits(bits.astype(np.uint8)).tobytes(), "big")

rows = []
for split, entry in splits.items():
    for p in sorted(entry["images"].iterdir()):
        if not (p.is_file() and p.suffix.lower() in IMG_EXT):
            continue
        rec = {"split": split, "file": p.name, "stem": p.stem, "path": str(p),
               "readable": True, "error": "", "w": 0, "h": 0, "md5": "", "dh": 0}
        try:
            with Image.open(p) as im:
                im.verify()                        # step 1: file header
            with Image.open(p) as im:
                im.load()                          # step 2: full decode (catches truncated downloads)
                rec["w"], rec["h"] = im.size
                rec["dh"] = dhash(im)
            rec["md5"] = hashlib.md5(p.read_bytes()).hexdigest()
        except Exception as e:
            rec["readable"], rec["error"] = False, f"{type(e).__name__}: {e}"
        rows.append(rec)

scan = pd.DataFrame(rows)
ok = scan[scan["readable"]].reset_index(drop=True)
print(f"Scanned {len(scan)} images · OK: {len(ok)} · corrupt: {len(scan) - len(ok)}")


## §4 — Corrupt files, duplicates, resolution

| Finding | Severity | Why |
|---|---|---|
| Corrupt file | 🔴 | Crashes training mid-run, sometimes after half an hour |
| Duplicates **across** splits | 🔴 | Data leak — the model saw validation early, score is inflated |
| Duplicates **within** a split | 🟡 | Image is counted twice, slightly skews training |
| Near-duplicates | 🟡/🔴 | Same issue, just harder to catch |


In [ ]:
from collections import defaultdict

# ── Corrupt ──────────────────────────────────────────────────────────────────
corrupt = scan[~scan["readable"]]
print(f"Corrupt files: {len(corrupt)}")

# ── Exact duplicates (MD5) ───────────────────────────────────────────────────
groups = defaultdict(list)
for r in ok.itertuples():
    groups[r.md5].append((r.split, r.file))
exact_dupes = {k: v for k, v in groups.items() if len(v) > 1}
cross_exact = {k: v for k, v in exact_dupes.items() if len({s for s, _ in v}) > 1}
print(f"Exact duplicates: {len(exact_dupes)} groups · of which cross-split: {len(cross_exact)}")

# ── Near-duplicates (dHash) ──────────────────────────────────────────────────
# Hamming distance between every pair: a@(1-b) + (1-a)@b  -> one small matrix instead of a nested loop
bits = np.array([[int(c) for c in format(v, "064b")] for v in ok["dh"]], dtype=np.int16)
inv  = 1 - bits
dist = bits @ inv.T + inv @ bits.T
np.fill_diagonal(dist, 99)

near_pairs = []
for i, j in zip(*np.where(np.triu(dist <= NEAR_DUP_BITS, k=1))):
    a, b = ok.iloc[i], ok.iloc[j]
    if a["md5"] == b["md5"]:
        continue                                   # already counted as an exact duplicate
    near_pairs.append((a["split"], a["file"], b["split"], b["file"],
                       int(dist[i, j]), a["split"] != b["split"]))
cross_near = [p for p in near_pairs if p[5]]
print(f"Near-duplicates (≤{NEAR_DUP_BITS} bits): {len(near_pairs)} pairs · "
      f"of which cross-split: {len(cross_near)}")

# ── Resolution ───────────────────────────────────────────────────────────────
low_res = ok[ok[["w", "h"]].min(axis=1) < MIN_SHORT_SIDE]
print(f"Images with short side < {MIN_SHORT_SIDE}px: {len(low_res)}")

sizes = ok.groupby(["w", "h"]).size().sort_values(ascending=False)
print(f"\nCommon sizes (out of {len(sizes)} distinct sizes):")
print(sizes.head(3).to_string())


## §5 — Label integrity

Walk every `.txt` line by line. YOLO format: `class_id x_center y_center width height`,
with all coordinates **normalized to 0..1** — so you can check out-of-bounds without opening the image.

**Why there is a tolerance (`BOX_EPS`):** Roboflow's export rounds coordinates, so a box that touches
the exact edge of the image sometimes comes out as `1.0000078` instead of `1.0` — an overshoot of a few
thousandths of a pixel, which looks perfect by eye and needs no fix. Without a tolerance every such box
would land on the action list as "broken label".

| Finding | Action |
|---|---|
| Box outside 0..1 by more than `BOX_EPS` / width or height 0 | 🔴 Fix in Annotate |
| `class_id` out of range | 🔴 Broken label |
| Line that is not 5 numbers | 🔴 Corrupt file |
| Label file with no image (orphan) | 🔴 Something broke in the export |
| Image with no label / empty label | 🟡 Check — either intentional background, or it was missed |


In [ ]:
label_problems = []          # (split, image_file, kind, detail)
orphan_labels  = []
empty_labels   = []
instances      = []          # for the distribution counts in §6

def add(split, img, kind, detail=""):
    label_problems.append((split, img, kind, detail))

for split, entry in splits.items():
    # from scan, not ok: a corrupt image can still have a valid label file, and it is not an "orphan"
    img_by_stem = {r.stem: r.file for r in scan.itertuples() if r.split == split}
    lbl_dir = entry["labels"]
    lbl_stems = set()

    for lp in sorted(lbl_dir.glob("*.txt")) if lbl_dir.is_dir() else []:
        lbl_stems.add(lp.stem)
        img = img_by_stem.get(lp.stem)
        if img is None:
            orphan_labels.append((split, lp.name))
            continue

        lines = [ln for ln in lp.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if not lines:
            empty_labels.append((split, img))
            continue

        for n, ln in enumerate(lines, 1):
            parts = ln.split()
            if len(parts) != 5:
                add(split, img, "malformed label line", f"line {n}: {len(parts)} values instead of 5")
                continue
            try:
                cid = int(float(parts[0]))
                x, y, w, h = map(float, parts[1:])
            except ValueError:
                add(split, img, "malformed label line", f"line {n}: non-numeric value")
                continue

            if not 0 <= cid < len(CLASS_NAMES):
                add(split, img, "class_id out of range",
                    f"line {n}: class_id={cid}, valid 0..{len(CLASS_NAMES) - 1}")
                continue
            if w <= 0 or h <= 0:
                add(split, img, "zero-size box", f"line {n}: w={w}, h={h}")
            else:
                # largest overshoot among the four edges. Up to BOX_EPS is export rounding
                # (sub-pixel, invisible) and not a labeling error — nothing to fix in Annotate.
                over = max(w / 2 - x, x + w / 2 - 1, h / 2 - y, y + h / 2 - 1)
                if over > BOX_EPS:
                    add(split, img, "box outside image bounds",
                        f"line {n}: {CLASS_NAMES[cid]} ({x:.3f},{y:.3f},{w:.3f},{h:.3f}) "
                        f"— overshoot of {over:.3f}")
            instances.append({"split": split, "class": CLASS_NAMES[cid], "file": img})

    for stem, img in img_by_stem.items():
        if stem not in lbl_stems:
            empty_labels.append((split, img))

print(f"Real label problems : {len(label_problems)}")
print(f"Orphan labels (no image): {len(orphan_labels)}")
print(f"Images with no/empty label: {len(empty_labels)}")
if label_problems:
    display(pd.DataFrame(label_problems,
                         columns=["split", "image", "issue", "detail"]).head(20))


## §6 — Numbers for the report

**The distinction to keep:** "how many images contain the class" ≠ "how many instances the class has".
One street image can contain 15 `person` boxes. The number that matters for training is **instances**.


In [ ]:
inst = pd.DataFrame(instances)
pivot = inst.pivot_table(index="class", columns="split", values="file",
                         aggfunc="count", fill_value=0)
for c in ("train", "valid", "test"):
    if c not in pivot.columns:
        pivot[c] = 0
pivot = pivot[["train", "valid", "test"]].reindex(CLASS_NAMES).fillna(0).astype(int)
pivot["instances"] = pivot.sum(axis=1)
pivot["images"] = inst.groupby("class")["file"].nunique().reindex(CLASS_NAMES).fillna(0).astype(int)
display(pivot)

ratio = pivot["instances"].max() / max(pivot["instances"].min(), 1)
print(f"Imbalance ratio most-common to rarest class: {ratio:.1f}:1", end="  ")
print("🟡 Worth mentioning in the report / consider class weights" if ratio > 3 else "✅ Reasonable")


## §7 — 🎯 Delete list file

**This is the deliverable.** Saved to `roboflow_delete_list.txt` in the notebook folder, grouped by reason, with filenames
as Roboflow knows them — without the `_jpg.rf.<hash>.jpg` suffix that the export adds,
because that suffix is exactly what stops search from finding them.


In [ ]:
import re
from collections import Counter

# ── Filename as Roboflow knows it ────────────────────────────────────────────
# The export appends _jpg.rf.<hash>.jpg to every file, so searching for the full
# local filename returns nothing in Roboflow. Restore the original name.
_RF_SUFFIX = re.compile(r"\.rf\.[0-9A-Za-z]+\.\w+$")
_RF_EXT    = re.compile(r"_(jpg|jpeg|png|bmp|webp)$", re.I)

def _core(f):
    return _RF_EXT.sub("", _RF_SUFFIX.sub("", f))

_dup_names = Counter(_core(f) for f in scan["file"])

def rf(f):
    """Name for Roboflow search. If two files share a source name — add a distinguishing hash."""
    c = _core(f)
    if _dup_names[c] > 1:
        return f"{c}   (copy {f.split('.rf.')[-1].split('.')[0][:6]})"
    return c

L = []
A = L.append

A("=" * 72)
A("ACTION LIST FOR ROBOFLOW")
A("Auto-generated by dataset_qc.ipynb")
A("=" * 72)
A("")
A(f"Dataset:  {PROJECT_URL}")
A("")
A("How to find an image:")
A("  Images (or Dataset)  ->  search field at the top  ->  paste the name from the list.")
A("  If search asks for a query:   filename:*ia_500000787*    (you can add  split:train)")
A("")
A("  Names here are source names as Roboflow knows them, without the _jpg.rf.<hash>.jpg")
A("  suffix the export adds — so searching the full local filename returns nothing.")
A("  The same name also finds the local file (search in the dataset folder).")
A("  A short name like 40 will return many results — search it with the extension:  40.jpg")
A('  "(copy XXXXXX)" = two different files with the same source name; tell them apart by the image itself.')
A("")
A("  After all deletes/fixes:  Generate  ->  new version  ->  send a new link to the team.")
A("")

def section(title, note=""):
    A("")
    A("-" * 72)
    A(title)
    if note:
        A("  " + note)
    A("-" * 72)

# ── [1] Corrupt ──────────────────────────────────────────────────────────────
section(f"[1] 🔴 DELETE — corrupt files ({len(corrupt)})",
        "File does not open at all. Will crash training mid-run.")
if len(corrupt) == 0:
    A("  ✅ None.")
for r in corrupt.itertuples():
    A(f"  [{r.split}]  {rf(r.file)}")
    A(f"           reason: {r.error}")

# ── [2] Exact duplicates ─────────────────────────────────────────────────────
section(f"[2] 🔴 DELETE — exact duplicates ({len(exact_dupes)} groups)",
        "Identical bit-by-bit. Keep one, delete the rest.")
if not exact_dupes:
    A("  ✅ None.")
for i, v in enumerate(exact_dupes.values(), 1):
    cross = " ⚠ cross-split — data leak!" if len({s for s, _ in v}) > 1 else ""
    A(f"  group {i}:{cross}")
    keep, *drop = sorted(v, key=lambda t: {"train": 0, "valid": 1, "test": 2}.get(t[0], 3))
    A(f"    ✔ keep    [{keep[0]}]  {rf(keep[1])}")
    for s, f in drop:
        A(f"    ✘ delete  [{s}]  {rf(f)}")

# ── [3] Near-duplicates ──────────────────────────────────────────────────────
section(f"[3] 🟡 REVIEW and delete one of each pair — near-duplicates ({len(near_pairs)} pairs)",
        "Look the same but the file differs. A cross-split pair is the serious problem.")
if not near_pairs:
    A("  ✅ None.")
for s1, f1, s2, f2, d, cross in sorted(near_pairs, key=lambda p: (not p[5], p[4])):
    A(f"  {'⚠ cross-split' if cross else '  same split  '} (diff {d} bits)")
    A(f"    [{s1}]  {rf(f1)}")
    A(f"    [{s2}]  {rf(f2)}")

# ── [4] Broken labels ────────────────────────────────────────────────────────
by_img = defaultdict(list)
for split, img, kind, detail in label_problems:
    by_img[(split, img)].append(f"{kind} — {detail}" if detail else kind)

section(f"[4] 🔴 FIX in Annotate (do not delete) — broken labels ({len(by_img)} images)",
        "Open the image in Roboflow > Annotate and fix the box.")
if not by_img:
    A("  ✅ None.")
for (split, img), probs in sorted(by_img.items()):
    A(f"  [{split}]  {rf(img)}")
    for p in probs:
        A(f"           {p}")

# ── [5] Orphan labels ────────────────────────────────────────────────────────
section(f"[5] 🔴 Label with no image ({len(orphan_labels)})",
        ".txt file with no image. Sign the export broke — re-export.")
if not orphan_labels:
    A("  ✅ None.")
for s, f in orphan_labels:
    A(f"  [{s}]  {rf(f)}")

# ── [6] No label ─────────────────────────────────────────────────────────────
section(f"[6] 🟡 REVIEW — images with no label ({len(empty_labels)})",
        "Either intentional background, or they were missed. If missed — the model explicitly learns 'nothing here'.")
if not empty_labels:
    A("  ✅ None.")
for s, f in sorted(empty_labels)[:200]:
    A(f"  [{s}]  {rf(f)}")
if len(empty_labels) > 200:
    A(f"  ... and {len(empty_labels) - 200} more. If the count is large — likely intentional export behavior.")

# ── [7] Low resolution ───────────────────────────────────────────────────────
section(f"[7] 🟡 REVIEW — low resolution ({len(low_res)})",
        f"Short side < {MIN_SHORT_SIDE}px. Usually junk that got in by mistake.")
if len(low_res) == 0:
    A("  ✅ None.")
for r in low_res.itertuples():
    A(f"  [{r.split}]  {rf(r.file)}   ({r.w}x{r.h})")

# ── Summary ──────────────────────────────────────────────────────────────────
n_delete = (len(corrupt)
            + sum(len(v) - 1 for v in exact_dupes.values()))
A("")
A("=" * 72)
A("Summary")
A("=" * 72)
A(f"  Delete for sure         : {n_delete}")
A(f"  Review and decide       : {len(near_pairs)} pairs + {len(low_res)} resolution + {len(empty_labels)} unlabeled")
A(f"  Fix in Annotate         : {len(by_img)}")
A(f"  Class order             : {'✅ OK' if CLASSES_OK else '🔴 does not match expected — align with the team'}")
A(f"  Total images in dataset : {len(scan)}")
A("")
A("After doing the work:  Generate a new version  ->  replace ROBOFLOW_URL in §0  ->  re-run to verify.")

OUT_TXT.write_text("\n".join(L), encoding="utf-8")
print(f"✅ Saved: {OUT_TXT.resolve()}")
print()
print("\n".join(L[-12:]))
